# Sensitivity analysis runner — no mutation, revised contract-period engine

This notebook runs no-mutation sensitivity and robustness cases around the frozen baseline using the external `verified_simulation_engine.py`.

Recommended sequence:

1. Confirm that `verified_simulation_engine.py` implements the revised estimator: contract-period exposure, standard deviations across replications, the contract-period As-Generated shortfall penalty, and the no-contract outside option.
2. Run one-factor-at-a-time sensitivity around the baseline.
3. Keep mutation / dependence perturbation in the separate mutation workflow.

This notebook:
- builds or reads a case table
- optionally creates case-specific transformed sample banks by scaling residuals around quarter × hour profiles
- runs the verified simulation engine once per enabled case on fixed saved sample pools
- writes a case manifest and run-status table under one sensitivity-output root

The revised main formulation does not use the legacy hourly penalty multiplier `gamma`. Legacy `GAMMA_*` rows are retained only for reproducing the earlier placeholder diagnostics and are disabled by default.

In [ ]:

from pathlib import Path
import importlib.util
import json
import re
import shutil
import traceback

import numpy as np
import pandas as pd


In [ ]:
# ============================================================
# Submission dtype helpers
# ============================================================
# These wrappers reduce DataFrame memory use without changing the financial
# calculations: identifiers/counters are downcast, repeated labels become
# categoricals, and continuous numerical columns remain float64.
import numpy as np

_PD_READ_CSV = pd.read_csv
_PD_READ_EXCEL = pd.read_excel

_INTEGER_DTYPE_CANDIDATES = {
    "match_id": np.int32,
    "hour": np.int16,
    "hour_index": np.int16,
    "replication": np.int16,
    "case_order": np.int16,
    "enabled": np.int8,
    "rank": np.int32,
}

_CATEGORY_DTYPE_CANDIDATES = {
    "case_id",
    "case_family",
    "case_label",
    "combined_category",
    "metric",
    "mutation_axis",
    "mutation_direction",
    "mutation_family",
    "mutation_label",
    "ppa_type",
    "profile_type",
    "risk_group",
    "risk_label",
    "scenario_name",
    "scenario_type",
    "solution_type",
    "status",
    "variable",
    "var_i",
    "var_j",
}


def _integer_dtype_fits(values, dtype) -> bool:
    if len(values) == 0:
        return True
    info = np.iinfo(dtype)
    return float(np.nanmin(values)) >= info.min and float(np.nanmax(values)) <= info.max


def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Conservatively compact non-financial columns after file loading."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df

    for col in df.columns:
        series = df[col]
        if pd.api.types.is_integer_dtype(series.dtype):
            df[col] = pd.to_numeric(series, downcast="integer")

    for col, dtype in _INTEGER_DTYPE_CANDIDATES.items():
        if col not in df.columns:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.isna().any():
            continue
        values = numeric.to_numpy(dtype="float64", copy=False)
        rounded = np.rint(values)
        if np.array_equal(values, rounded) and _integer_dtype_fits(rounded, dtype):
            df[col] = rounded.astype(dtype, copy=False)

    n_rows = len(df)
    for col in _CATEGORY_DTYPE_CANDIDATES.intersection(df.columns):
        series = df[col]
        if pd.api.types.is_categorical_dtype(series.dtype):
            continue
        if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue
        n_unique = int(non_null.nunique())
        if n_unique <= min(128, max(2, n_rows // 2)):
            df[col] = series.astype("category")

    return df


def read_csv_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_CSV(*args, **kwargs))


def read_excel_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_EXCEL(*args, **kwargs))


In [ ]:
# ============================================================
# Control panel
# ============================================================
CONFIG = {
    # Optional explicit local workplace root. Leave as None when running from the
    # Code_Submission workplace or from the sensitivity bundle inside Code_Submission.
    # The resolver is intentionally local: it checks only the sensitivity bundle
    # and the inferred/explicit workplace root, not arbitrary parent folders.
    "workplace_dir": None,

    # Core engine built from the revised formulas + verify notebook.
    # Keep this file in the same sensitivity bundle folder as this notebook, or
    # in a direct child folder of the local workplace root.
    "verified_engine_file": "verified_simulation_engine.py",

    # Inputs. Relative paths are resolved inside the local workplace root, with
    # support for the common sibling folders "Input data and files" and
    # "Data and files".
    "baseline_sample_root": "Simplified baseline realized samples",
    "match_folder_name": "Match files of original data",

    # Sensitivity outputs. Relative paths are created inside the sensitivity bundle.
    "sensitivity_output_root": "Output files (Sensitivity, No Mutation, Verified)",
    "case_table_csv": "Sensitivity_Cases_Verified_template.csv",

    # Case-table behavior
    "build_default_case_table_if_missing": True,
    "run_only_enabled_cases": True,

    # Revised-formulation behavior. Keep False for final revised-model runs.
    # Set True only to reproduce the retained legacy gamma diagnostics.
    "run_legacy_gamma_cases": False,

    # Existing-output behavior
    "reuse_existing_case_outputs": False,
    "rebuild_case_sample_banks": False,

    # Verified-engine controls
    "selected_match_ids": None,        # Example: [1, 2, 3]
    "save_hourly_fe": False,
    "save_best_fe_even_if_infeasible": False,
    "save_per_match_grid": False,
    "save_timestamp_in_hourly_fe": False,

    # Runner controls
    "stop_on_error": False,
    "print_case_progress": True,
}
CONFIG

In [ ]:
DEFAULT_CASES = pd.DataFrame([
    {"case_id":"BASE","enabled":1,"case_order":0,"case_family":"Baseline","case_label":"Baseline","case_value":"1.0","lambda_s":0.0,"lambda_b":0.0,"gamma":1.0,"volume_resid_multiplier":1.0,"price_resid_multiplier":1.0,"notes":"Reference case"},
    {"case_id":"LS_0253","enabled":1,"case_order":10,"case_family":"SellerRisk","case_label":"lambda_s = 0.253","case_value":"0.253","lambda_s":0.253,"lambda_b":0.0,"gamma":1.0,"volume_resid_multiplier":1.0,"price_resid_multiplier":1.0,"notes":"Seller risk only"},
    {"case_id":"LS_0524","enabled":1,"case_order":11,"case_family":"SellerRisk","case_label":"lambda_s = 0.524","case_value":"0.524","lambda_s":0.524,"lambda_b":0.0,"gamma":1.0,"volume_resid_multiplier":1.0,"price_resid_multiplier":1.0,"notes":"Seller risk only"},
    {"case_id":"LS_0842","enabled":1,"case_order":12,"case_family":"SellerRisk","case_label":"lambda_s = 0.842","case_value":"0.842","lambda_s":0.842,"lambda_b":0.0,"gamma":1.0,"volume_resid_multiplier":1.0,"price_resid_multiplier":1.0,"notes":"Seller risk only"},
    {"case_id":"LB_0253","enabled":1,"case_order":20,"case_family":"BuyerRisk","case_label":"lambda_b = 0.253","case_value":"0.253","lambda_s":0.0,"lambda_b":0.253,"gamma":1.0,"volume_resid_multiplier":1.0,"price_resid_multiplier":1.0,"notes":"Buyer risk only"},
    {"case_id":"LB_0524","enabled":1,"case_order":21,"case_family":"BuyerRisk","case_label":"lambda_b = 0.524","case_value":"0.524","lambda_s":0.0,"lambda_b":0.524,"gamma":1.0,"volume_resid_multiplier":1.0,"price_resid_multiplier":1.0,"notes":"Buyer risk only"},
    {"case_id":"LB_0842","enabled":1,"case_order":22,"case_family":"BuyerRisk","case_label":"lambda_b = 0.842","case_value":"0.842","lambda_s":0.0,"lambda_b":0.842,"gamma":1.0,"volume_resid_multiplier":1.0,"price_resid_multiplier":1.0,"notes":"Buyer risk only"},
    {"case_id":"BOTH_0253","enabled":1,"case_order":30,"case_family":"BothRisk","case_label":"lambda_s = lambda_b = 0.253","case_value":"0.253","lambda_s":0.253,"lambda_b":0.253,"gamma":1.0,"volume_resid_multiplier":1.0,"price_resid_multiplier":1.0,"notes":"Both parties risk-sensitive"},
    {"case_id":"BOTH_0524","enabled":1,"case_order":31,"case_family":"BothRisk","case_label":"lambda_s = lambda_b = 0.524","case_value":"0.524","lambda_s":0.524,"lambda_b":0.524,"gamma":1.0,"volume_resid_multiplier":1.0,"price_resid_multiplier":1.0,"notes":"Both parties risk-sensitive"},
    {"case_id":"GAMMA_075","enabled":0,"case_order":40,"case_family":"LegacyGamma","case_label":"legacy gamma = 0.75","case_value":"0.75","lambda_s":0.0,"lambda_b":0.0,"gamma":0.75,"volume_resid_multiplier":1.0,"price_resid_multiplier":1.0,"notes":"Legacy hourly-penalty sensitivity only; disabled under revised main formulation"},
    {"case_id":"GAMMA_125","enabled":0,"case_order":41,"case_family":"LegacyGamma","case_label":"legacy gamma = 1.25","case_value":"1.25","lambda_s":0.0,"lambda_b":0.0,"gamma":1.25,"volume_resid_multiplier":1.0,"price_resid_multiplier":1.0,"notes":"Legacy hourly-penalty sensitivity only; disabled under revised main formulation"},
    {"case_id":"GAMMA_150","enabled":0,"case_order":42,"case_family":"LegacyGamma","case_label":"legacy gamma = 1.50","case_value":"1.50","lambda_s":0.0,"lambda_b":0.0,"gamma":1.50,"volume_resid_multiplier":1.0,"price_resid_multiplier":1.0,"notes":"Legacy hourly-penalty sensitivity only; disabled under revised main formulation"},
    {"case_id":"VOL_075","enabled":1,"case_order":50,"case_family":"VolumeVol","case_label":"volume residual × 0.75","case_value":"0.75","lambda_s":0.0,"lambda_b":0.0,"gamma":1.0,"volume_resid_multiplier":0.75,"price_resid_multiplier":1.0,"notes":"Lower generation/demand residual scale"},
    {"case_id":"VOL_125","enabled":1,"case_order":51,"case_family":"VolumeVol","case_label":"volume residual × 1.25","case_value":"1.25","lambda_s":0.0,"lambda_b":0.0,"gamma":1.0,"volume_resid_multiplier":1.25,"price_resid_multiplier":1.0,"notes":"Higher generation/demand residual scale"},
    {"case_id":"PRICE_075","enabled":1,"case_order":60,"case_family":"PriceVol","case_label":"price residual × 0.75","case_value":"0.75","lambda_s":0.0,"lambda_b":0.0,"gamma":1.0,"volume_resid_multiplier":1.0,"price_resid_multiplier":0.75,"notes":"Lower price residual scale"},
    {"case_id":"PRICE_125","enabled":1,"case_order":61,"case_family":"PriceVol","case_label":"price residual × 1.25","case_value":"1.25","lambda_s":0.0,"lambda_b":0.0,"gamma":1.0,"volume_resid_multiplier":1.0,"price_resid_multiplier":1.25,"notes":"Higher price residual scale"},
])
DEFAULT_CASES

In [ ]:

# ============================================================
# Helpers
# ============================================================
def unique_paths(paths):
    out = []
    seen = set()
    for path in paths:
        if path is None:
            continue
        p = Path(path).expanduser()
        try:
            key = str(p.resolve()) if p.exists() else str(p)
        except Exception:
            key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


def candidate_roots(*anchors, max_parent_depth: int = 0):
    """Return local search roots.

    The default no longer walks upward through arbitrary parent folders. This keeps
    the runner anchored to the sensitivity bundle and the local workspace instead
    of accidentally reading data from another project folder.
    """
    roots = []
    for anchor in anchors:
        if anchor is None:
            continue
        p = Path(anchor).expanduser()
        if p.suffix:
            p = p.parent
        roots.append(p)
        if max_parent_depth > 0:
            roots.extend(list(p.parents)[:max_parent_depth])
    return unique_paths(roots)


def infer_workplace_root(bundle_root: Path, notebook_cwd: Path, explicit_workplace=None) -> Path:
    """Infer the local Code_Submission/workplace root without searching outside it."""
    if explicit_workplace:
        return Path(explicit_workplace).expanduser().resolve()

    candidates = unique_paths([
        bundle_root,
        notebook_cwd,
        bundle_root.parent,
        notebook_cwd.parent,
    ])
    for cand in candidates:
        if (cand / "Input data and files").exists() or (cand / "Data and files").exists():
            return cand.resolve()
    # Conservative fallback: if the engine lives in a named sensitivity bundle,
    # its parent is normally the Code_Submission workplace; otherwise use the bundle.
    if bundle_root.name.lower() in {"sensitivity_no_mutation", "sensitivity no mutation"}:
        return bundle_root.parent.resolve()
    return bundle_root.resolve()


def resolve_existing_dir_flexible(raw_path, *anchors, include_child_level: bool = False):
    checked = []
    p = Path(raw_path).expanduser()

    def dir_aliases(root: Path):
        aliases = [root]
        aliases.append(root / "Input data and files")
        aliases.append(root / "Data and files")
        return unique_paths(aliases)

    if p.is_absolute():
        checked.append(p)
        if p.exists() and p.is_dir():
            return p.resolve()
    else:
        roots = candidate_roots(*anchors)
        for root in roots:
            for alias in dir_aliases(root):
                cand = (alias / p)
                checked.append(cand)
                if cand.exists() and cand.is_dir():
                    return cand.resolve()
        if include_child_level:
            for root in roots:
                for alias in dir_aliases(root):
                    if not alias.exists() or not alias.is_dir():
                        continue
                    for child in alias.iterdir():
                        if child.is_dir():
                            cand = child / p
                            checked.append(cand)
                            if cand.exists() and cand.is_dir():
                                return cand.resolve()
    raise FileNotFoundError(
        "Could not resolve an existing directory. Tried:\n" + "\n".join(str(x) for x in unique_paths(checked))
    )


def resolve_existing_file_flexible(raw_path, *anchors, include_child_level: bool = False):
    checked = []
    p = Path(raw_path).expanduser()

    def dir_aliases(root: Path):
        aliases = [root]
        aliases.append(root / "Input data and files")
        aliases.append(root / "Data and files")
        return unique_paths(aliases)

    if p.is_absolute():
        checked.append(p)
        if p.exists() and p.is_file():
            return p.resolve()
    else:
        roots = candidate_roots(*anchors)
        for root in roots:
            for alias in dir_aliases(root):
                cand = (alias / p)
                checked.append(cand)
                if cand.exists() and cand.is_file():
                    return cand.resolve()
        if include_child_level:
            for root in roots:
                for alias in dir_aliases(root):
                    if not alias.exists() or not alias.is_dir():
                        continue
                    for child in alias.iterdir():
                        if child.is_dir():
                            cand = child / p
                            checked.append(cand)
                            if cand.exists() and cand.is_file():
                                return cand.resolve()
    raise FileNotFoundError(
        "Could not resolve an existing file. Tried:\n" + "\n".join(str(x) for x in unique_paths(checked))
    )


def prepare_relative_path(raw_path, anchor_dir: Path) -> Path:
    p = Path(raw_path).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (anchor_dir / p).resolve()


def slugify(text):
    text = str(text).strip()
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    return text.strip("_") or "case"


def extract_match_id(path_like):
    matched = re.search(r"(\d+)", Path(path_like).stem)
    if not matched:
        raise ValueError(f"Could not parse match_id from: {path_like}")
    return int(matched.group(1))


def discover_replication_dirs(sample_root: Path):
    rep_dirs = sorted([p for p in sample_root.iterdir() if p.is_dir() and p.name.lower().startswith("rep_")])
    if rep_dirs:
        return rep_dirs
    flat_csvs = list(sample_root.glob("*.csv"))
    if flat_csvs:
        return [sample_root]
    raise FileNotFoundError(f"No replication folders or flat csv files found under {sample_root}")


def ensure_case_table(case_table_path: Path, build_if_missing: bool = True) -> pd.DataFrame:
    if case_table_path.exists():
        return read_csv_optimized(case_table_path)
    if not build_if_missing:
        raise FileNotFoundError(f"Case table not found: {case_table_path}")
    case_table_path.parent.mkdir(parents=True, exist_ok=True)
    DEFAULT_CASES.to_csv(case_table_path, index=False)
    return DEFAULT_CASES.copy()


def import_verified_engine(engine_path: Path):
    spec = importlib.util.spec_from_file_location("verified_simulation_engine", engine_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def compute_match_profile_tables(sample_root: Path):
    rep_dirs = discover_replication_dirs(sample_root)
    profiles = {}
    match_ids = sorted({extract_match_id(p.name) for rep in rep_dirs for p in rep.glob("*.csv")})
    for match_id in match_ids:
        parts = []
        for rep in rep_dirs:
            fp = rep / f"{match_id:03d}.csv"
            if fp.exists():
                df = read_csv_optimized(fp)
                if not {"hour", "quarter", "generation", "demand", "seller_lmp", "buyer_lmp_out", "buyer_lmp_in"}.issubset(df.columns):
                    raise KeyError(f"Sample file is missing required columns: {fp}")
                parts.append(df[["quarter", "hour", "generation", "demand", "seller_lmp", "buyer_lmp_out", "buyer_lmp_in"]].copy())
        if not parts:
            continue
        all_df = pd.concat(parts, ignore_index=True)
        prof = (
            all_df.groupby(["quarter", "hour"], dropna=False, observed=False)
            .agg({
                "generation": "mean",
                "demand": "mean",
                "seller_lmp": "mean",
                "buyer_lmp_out": "mean",
                "buyer_lmp_in": "mean",
            })
            .reset_index()
            .rename(columns={
                "generation": "profile_generation",
                "demand": "profile_demand",
                "seller_lmp": "profile_seller_lmp",
                "buyer_lmp_out": "profile_buyer_lmp_out",
                "buyer_lmp_in": "profile_buyer_lmp_in",
            })
        )
        profiles[match_id] = prof
    return profiles


def apply_case_transform(df: pd.DataFrame,
                         profile_df: pd.DataFrame,
                         volume_resid_multiplier: float,
                         price_resid_multiplier: float) -> pd.DataFrame:
    out = df.copy()
    merged = out.merge(profile_df, on=["quarter", "hour"], how="left")

    if not np.isclose(volume_resid_multiplier, 1.0):
        out["generation"] = (
            merged["profile_generation"] +
            volume_resid_multiplier * (merged["generation"] - merged["profile_generation"])
        ).clip(lower=0.0)
        out["demand"] = (
            merged["profile_demand"] +
            volume_resid_multiplier * (merged["demand"] - merged["profile_demand"])
        ).clip(lower=0.0)

    if not np.isclose(price_resid_multiplier, 1.0):
        out["seller_lmp"] = (
            merged["profile_seller_lmp"] +
            price_resid_multiplier * (merged["seller_lmp"] - merged["profile_seller_lmp"])
        )
        out["buyer_lmp_out"] = (
            merged["profile_buyer_lmp_out"] +
            price_resid_multiplier * (merged["buyer_lmp_out"] - merged["profile_buyer_lmp_out"])
        )

        # Preserve the original purchase-side wedge in the no-mutation setting.
        wedge = df["buyer_lmp_in"] - df["buyer_lmp_out"]
        out["buyer_lmp_in"] = out["buyer_lmp_out"] + wedge

    return out


def build_case_sample_bank(case_row: pd.Series,
                           baseline_sample_root: Path,
                           case_sample_root: Path,
                           profile_cache: dict,
                           rebuild: bool = False) -> Path:
    volume_mult = float(case_row["volume_resid_multiplier"])
    price_mult = float(case_row["price_resid_multiplier"])

    if np.isclose(volume_mult, 1.0) and np.isclose(price_mult, 1.0):
        return baseline_sample_root

    if case_sample_root.exists() and not rebuild:
        return case_sample_root.resolve()

    if case_sample_root.exists():
        shutil.rmtree(case_sample_root)
    case_sample_root.mkdir(parents=True, exist_ok=True)

    for rep_dir in discover_replication_dirs(baseline_sample_root):
        target_rep_dir = case_sample_root / rep_dir.name
        target_rep_dir.mkdir(parents=True, exist_ok=True)
        for src_file in sorted(rep_dir.glob("*.csv")):
            match_id = extract_match_id(src_file.name)
            profile_df = profile_cache.get(match_id)
            if profile_df is None:
                raise KeyError(f"Missing profile table for match_id={match_id}")
            df = read_csv_optimized(src_file)
            transformed = apply_case_transform(df, profile_df, volume_mult, price_mult)
            transformed.to_csv(target_rep_dir / src_file.name, index=False)

    return case_sample_root.resolve()


In [ ]:
# ============================================================
# Resolve paths and load inputs
# ============================================================
NOTEBOOK_CWD = Path.cwd().resolve()
EXPLICIT_WORKPLACE = CONFIG.get("workplace_dir")

# First resolve the engine from the local cwd / explicit workspace, allowing one
# direct child level so Code_Submission/Sensitivity_No_Mutation/verified_simulation_engine.py
# is found when the notebook is launched from Code_Submission.
pre_engine_anchors = unique_paths([
    NOTEBOOK_CWD,
    Path(EXPLICIT_WORKPLACE).expanduser() if EXPLICIT_WORKPLACE else None,
])
PRE_ENGINE_ROOTS = candidate_roots(*pre_engine_anchors)

engine_path = resolve_existing_file_flexible(
    CONFIG["verified_engine_file"],
    *PRE_ENGINE_ROOTS,
    include_child_level=True,
)

BUNDLE_ROOT = engine_path.parent.resolve()
WORKPLACE_ROOT = infer_workplace_root(
    bundle_root=BUNDLE_ROOT,
    notebook_cwd=NOTEBOOK_CWD,
    explicit_workplace=EXPLICIT_WORKPLACE,
)
SEARCH_ROOTS = unique_paths([BUNDLE_ROOT, WORKPLACE_ROOT])

baseline_sample_root = resolve_existing_dir_flexible(
    CONFIG["baseline_sample_root"],
    *SEARCH_ROOTS,
    include_child_level=True,
)

match_folder_path = resolve_existing_dir_flexible(
    CONFIG["match_folder_name"],
    *SEARCH_ROOTS,
    include_child_level=True,
)

case_table_path = prepare_relative_path(CONFIG["case_table_csv"], anchor_dir=BUNDLE_ROOT)
sensitivity_output_root = prepare_relative_path(CONFIG["sensitivity_output_root"], anchor_dir=BUNDLE_ROOT)

sensitivity_output_root.mkdir(parents=True, exist_ok=True)
(sensitivity_output_root / "Cases").mkdir(parents=True, exist_ok=True)
(sensitivity_output_root / "Case_Sample_Banks").mkdir(parents=True, exist_ok=True)

case_table_df = ensure_case_table(
    case_table_path=case_table_path,
    build_if_missing=CONFIG["build_default_case_table_if_missing"],
)

# The revised main formulation no longer varies the legacy hourly gamma penalty.
# If an older case table already has GAMMA_* rows enabled, disable them in memory
# unless the user explicitly requests legacy reproduction.
if not CONFIG.get("run_legacy_gamma_cases", False):
    legacy_mask = (
        case_table_df.get("case_id", pd.Series(index=case_table_df.index, dtype=str)).astype(str).str.upper().str.startswith("GAMMA_") |
        case_table_df.get("case_family", pd.Series(index=case_table_df.index, dtype=str)).astype(str).str.contains("Gamma", case=False, na=False)
    )
    if legacy_mask.any():
        case_table_df.loc[legacy_mask, "enabled"] = 0
        print(f"Legacy gamma cases disabled for revised-formulation run: {int(legacy_mask.sum())}")

if CONFIG["run_only_enabled_cases"] and "enabled" in case_table_df.columns:
    case_table_df = case_table_df.loc[case_table_df["enabled"].fillna(1).astype(int) == 1].copy()

case_table_df = case_table_df.sort_values(["case_order", "case_id"]).reset_index(drop=True)

print("Workspace root  :", WORKPLACE_ROOT)
print("Bundle root     :", BUNDLE_ROOT)
print("Verified engine :", engine_path)
print("Baseline samples:", baseline_sample_root)
print("Match folder    :", match_folder_path)
print("Output root     :", sensitivity_output_root)
print("Case table      :", case_table_path)
print(f"Cases selected  : {len(case_table_df)}")

engine = import_verified_engine(engine_path)
case_table_df.head()

In [ ]:

# Build profile cache only if at least one case needs sample-bank transformation.
needs_case_sample_transform = bool(
    (~np.isclose(pd.to_numeric(case_table_df["volume_resid_multiplier"], errors="coerce"), 1.0)).any() or
    (~np.isclose(pd.to_numeric(case_table_df["price_resid_multiplier"], errors="coerce"), 1.0)).any()
)

profile_cache = {}
if needs_case_sample_transform:
    profile_cache = compute_match_profile_tables(baseline_sample_root)
    print(f"Profile cache built for {len(profile_cache)} matches.")
else:
    print("No case requires transformed sample banks; baseline samples will be reused for every case.")


In [ ]:

# ============================================================
# Run all cases
# ============================================================
manifest_rows = []
run_status_rows = []

for _, case_row in case_table_df.iterrows():
    case_id = str(case_row["case_id"])
    case_label = str(case_row.get("case_label", case_id))
    case_slug = f"{slugify(case_id)}__{slugify(case_label)}"
    case_output_dir = sensitivity_output_root / "Cases" / case_slug
    case_sample_root = sensitivity_output_root / "Case_Sample_Banks" / case_slug

    result_file = case_output_dir / "Simulation_Best_Solutions_All_Matches.csv"
    case_uses_transformed_samples = not (
        np.isclose(float(case_row["volume_resid_multiplier"]), 1.0) and
        np.isclose(float(case_row["price_resid_multiplier"]), 1.0)
    )

    if CONFIG["print_case_progress"]:
        print()
        print("=" * 88)
        print(f"Case {case_id}: {case_label}")
        print("=" * 88)

    try:
        active_sample_root = build_case_sample_bank(
            case_row=case_row,
            baseline_sample_root=baseline_sample_root,
            case_sample_root=case_sample_root,
            profile_cache=profile_cache,
            rebuild=CONFIG["rebuild_case_sample_banks"],
        )

        if CONFIG["reuse_existing_case_outputs"] and result_file.exists():
            run_status = "skipped_existing_output"
            err_msg = ""
        else:
            overrides = {
                "sample_root_dir": str(active_sample_root),
                "match_folder_name": str(match_folder_path),
                "output_dir": str(case_output_dir),
                "lambda_s": float(case_row["lambda_s"]),
                "lambda_b": float(case_row["lambda_b"]),
                "gamma": float(case_row["gamma"]),
                "selected_match_ids": CONFIG["selected_match_ids"],
                "save_hourly_fe": bool(CONFIG["save_hourly_fe"]),
                "save_best_fe_even_if_infeasible": bool(CONFIG["save_best_fe_even_if_infeasible"]),
                "save_per_match_grid": bool(CONFIG["save_per_match_grid"]),
                "save_timestamp_in_hourly_fe": bool(CONFIG["save_timestamp_in_hourly_fe"]),
                "scenario_name": f"Sensitivity__{case_slug}",
            }
            engine.run_simulation(overrides)
            run_status = "success"
            err_msg = ""

        manifest_rows.append({
            "case_id": case_id,
            "case_label": case_label,
            "case_slug": case_slug,
            "case_family": case_row.get("case_family", ""),
            "case_order": case_row.get("case_order", np.nan),
            "case_value": case_row.get("case_value", ""),
            "lambda_s": case_row.get("lambda_s", np.nan),
            "lambda_b": case_row.get("lambda_b", np.nan),
            "gamma": case_row.get("gamma", np.nan),
            "volume_resid_multiplier": case_row.get("volume_resid_multiplier", np.nan),
            "price_resid_multiplier": case_row.get("price_resid_multiplier", np.nan),
            "uses_transformed_samples": case_uses_transformed_samples,
            "sample_root_dir": str(active_sample_root),
            "case_output_dir": str(case_output_dir),
            "status": run_status,
            "notes": case_row.get("notes", ""),
        })
        run_status_rows.append({
            "case_id": case_id,
            "case_label": case_label,
            "status": run_status,
            "error_message": err_msg,
        })

    except Exception as exc:
        tb = traceback.format_exc(limit=10)
        manifest_rows.append({
            "case_id": case_id,
            "case_label": case_label,
            "case_slug": case_slug,
            "case_family": case_row.get("case_family", ""),
            "case_order": case_row.get("case_order", np.nan),
            "case_value": case_row.get("case_value", ""),
            "lambda_s": case_row.get("lambda_s", np.nan),
            "lambda_b": case_row.get("lambda_b", np.nan),
            "gamma": case_row.get("gamma", np.nan),
            "volume_resid_multiplier": case_row.get("volume_resid_multiplier", np.nan),
            "price_resid_multiplier": case_row.get("price_resid_multiplier", np.nan),
            "uses_transformed_samples": case_uses_transformed_samples,
            "sample_root_dir": str(case_sample_root if case_uses_transformed_samples else baseline_sample_root),
            "case_output_dir": str(case_output_dir),
            "status": "failed",
            "notes": case_row.get("notes", ""),
        })
        run_status_rows.append({
            "case_id": case_id,
            "case_label": case_label,
            "status": "failed",
            "error_message": str(exc),
            "traceback": tb,
        })
        print(tb)
        if CONFIG["stop_on_error"]:
            raise

manifest_df = pd.DataFrame(manifest_rows).sort_values(["case_order", "case_id"]).reset_index(drop=True)
run_status_df = pd.DataFrame(run_status_rows).sort_values(["case_label", "case_id"]).reset_index(drop=True)

manifest_path = sensitivity_output_root / "Sensitivity_Case_Manifest.csv"
run_status_path = sensitivity_output_root / "Sensitivity_Case_Run_Status.csv"

manifest_df.to_csv(manifest_path, index=False)
run_status_df.to_csv(run_status_path, index=False)

print()
print("Saved:")
print(" ", manifest_path)
print(" ", run_status_path)

run_status_df


### Notes

- `volume_resid_multiplier` scales **generation** and **demand** residuals around the quarter × hour profile.
- `price_resid_multiplier` scales **seller_lmp** and **buyer_lmp_out** residuals around the quarter × hour profile.
- `buyer_lmp_in` is then rebuilt as `buyer_lmp_out_new + original_spread`, so the purchase-side wedge remains attached to the transformed nodal price.
- `GAMMA_*` cases are legacy diagnostics from the earlier hourly-penalty implementation and are disabled by default. The revised main formulation uses the contract-period As-Generated penalty handled by the simulation engine.

That keeps the no-mutation sensitivity step aligned with the revised contract-period formulation: saved scenario pools are reused, residual scaling changes dispersion around the quarter × hour profile, and risk preferences change only the objective/participation rule.